# Tutorial 4: Some example of Genetic Algorithms using DEAP
---
Faculty of Mathematics, Vienna, April 29, 2026.

Author: Sonia Foschiatti

---

### Learning Objectives
In this notebook we are going to view an implementation of the knapsack 0/1 problem and its resolution using Genetic Algorithms and the package DEAP. 

We will first create a knapsack class and then implement the Genetic Algorithm to solve the problem.

---
## 0. Setup and Imports

In [ ]:
# Uncomment this cell to install the libraries
#!pip install numpy matplotlib deap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from deap import base, tools, creator, algorithms

# fix a seed to reproduce the solution
SEED = 42
random.seed(SEED)

## 1. The Knapsack Problem

### What is the Knapsack Problem?

Immagine to have a fixed-size knapsack and you are given a set of items, each with a weight and a value. The knapsack problem consists in determining the number of items in the knapsack so that the total weight is less than or equal to a certain capacity and the total value is as large as possible.

### Mathematical Formulation: the 0-1 knapsack problem
In this model, we consider $n$ items, each item $i$ has a weight $w_i$ and a value $v_i$. The maximum weight capacity is fixed at the value $W$.
Let $x_i$ be a variable that takes the value 1 if the object is chosen to be put in the rucksack and 0 if it has to be discarded.

The goal is to maximize the function
$$f(x) = \sum_{i=1}^{n} v_i x_i$$
subject to the constraints:
$$\sum_{i=1}^{n} w_i x_i \leq W\qquad \text{and} \qquad x_i\in \{0,1\}.$$

**Remark:** This is an NP-hard problem, which means that exact solutions are computationally expensive for large $n$.

---
## 2. Creating the Knapsack Problem Class

Let's build a class to represent our specific knapsack problem instance.

1. **init function**: initialize an empty knapsack with items and maximum capacity.
2. **len function**: returns the number of available items.
3. **initData**: initialize the problem data inserting a list of items and the maximum capacity. Each item is a tuple given by a label, a weight and a value.
4. **getTotalValueFunction**: calculate the total value of the selected items
5. **printItems**: print the selected items, their weights and values

In [ ]:
class Knapsack:
    """
    A class representing the 0/1 Knapsack Problem.
    """
    
    def __init__(self):
        self.maxCapacity = 0
        self.items = []
        self.__initData()
    
    def __len__(self):
        return len(self.items)
    
    def __initData(self):
        self.items = [
            ("map", 9, 150),
            ("compass", 13, 35),
            ("water", 153, 200),
            ("sandwich", 50, 160),
            ("glucose", 15, 60),
            ("tin", 68, 45),
            ("banana", 27, 60),
            ("apple", 39, 40),
            ("cheese", 23, 30),
            ("suntan cream", 11, 70),
            ("camera", 32, 30),
            ("t-shirt", 24, 15),
            ("trousers", 48, 10),
            ("umbrella", 73, 40),
            ("waterproof trousers", 42, 70),
            ("waterproof overclothes", 43, 75),
            ("note-case", 22, 80),
            ("sunglasses", 7, 20),
            ("towel", 18, 12),
            ("socks", 4, 50),
            ("book", 30, 10)
        ]
        
        # Maximum weight capacity of the knapsack
        self.maxCapacity = 400
    
    def getTotalValue(self, zeroOneList):
        """
        Calculate the total value of selected items.
        
        Args:
            zeroOneList: A list of 0/1 values indicating which items are selected
                        (1 = selected, 0 = not selected)
        
        Returns:
            Total value of feasible items (items that fit in the capacity)
        """
        totalWeight = 0
        totalValue = 0
        
        for i in range(len(zeroOneList)):
            item_name, weight, value = self.items[i]
            
            # Only add item if it's selected (1) and fits in remaining capacity
            if zeroOneList[i] == 1:
                totalWeight += weight
                totalValue += value
        if totalWeight > self.maxCapacity:
            return 0
        
        return totalValue
    
    def printItems(self, zeroOneList):
        """
        Print the selected items and their cumulative weight/value.
        
        Args:
            zeroOneList: A list of 0/1 values indicating which items are selected
        """
        totalWeight = 0
        totalValue = 0
        
        print("\nSelected items:")
        print("-" * 80)
        
        for i in range(len(zeroOneList)):
            item_name, weight, value = self.items[i]
            
            # Only add item if it's selected and fits
            if zeroOneList[i] ==1 and totalWeight + weight <= self.maxCapacity:
                totalWeight += weight
                totalValue += value
                print(f"  {item_name:25s} | weight: {weight:3d} | value: {value:3d} | "
                      f"total weight: {totalWeight:3d} | total value: {totalValue:3d}")
        
        print("-" * 80)
        print(f"TOTAL: Weight = {totalWeight}/{self.maxCapacity}, Value = {totalValue}")
        print()

In [ ]:
# Create a knapsack class
knapsack = Knapsack()

print(f"Number of items available: {len(knapsack)}")
print(f"Maximum capacity: {knapsack.maxCapacity}")
print(f"\nFirst 5 items:")
for i in range(5):
    print(f"  {i+1}. Object: {knapsack.items[i][0]} | weight={knapsack.items[i][1]} | value={knapsack.items[i][2]}")

In [ ]:
# Generate a random solution (randomly select items)
def randomSolution(lengthKnapsack):
    return [random.randint(0,1) for _ in range(lengthKnapsack)]

randomSolution=randomSolution(len(knapsack))

print("Random solution (1=selected, 0=not selected):")
print(randomSolution)

knapsack.printItems(randomSolution)
print(f"Total value (if it is zero, then it is not a feasible solution): {knapsack.getTotalValue(randomSolution)}")

## 3. Setting Up the Genetic Algorithm using DEAP

Let's configure the parameters for our genetic algorithm.

In [ ]:
# Problem-specific parameters
items_length = len(knapsack)  # Number of items

# Genetic algorithm parameters
population_size = 200      # Number of individuals in each generation
pcross = 0.6          # Probability of crossover (60%)
pmutation = 0.2           # Probability of mutation (20%)
maxGenerations = 50       # Number of generations to evolve
HALL_OF_FAME_SIZE = 10     # Keep track of the best solutions

print("Genetic Algorithm Configuration:")
print(f"  Population size: {population_size}")
print(f"  Length of the chromosomes: {items_length}")
print(f"  Crossover probability: {pcross}")
print(f"  Mutation probability: {pmutation}")
print(f"  Maximum generations: {maxGenerations}")
print(f"  Chromosome length: {items_length} bits (one per item)")

### Step 1: Create the Toolbox

We create a toolbox using `Toolbox()`, a "container" of genetic operators. We can find it in the module `base`.

Then we register the `zeroOrOne` function using the Deap sintax:

`toolbox.register("function_name", function, arguments)`.

In [ ]:
# Create a toolbox for holding our genetic operators
toolbox = base.Toolbox()

# Register a function to create random 0 or 1 values
toolbox.register('zeroOrOne', random.randint, 0, 1)

print("Test our 'zeroOrOne' function: random bits =", [toolbox.zeroOrOne() for _ in range(10)])

### Step 2: Define Fitness and Individual

We need to tell DEAP:
- What "fitness" means (we want to maximize value). 
- What an "individual" looks like (a list of 0/1 values).

Remark: we define the actual Fitness function later!

In [ ]:
# Create a fitness class for maximization
# weights=(1.0,) : maximizing a single objective
# weights=(-1.0,) : minimizing a single objective
creator.create('FitnessMax', base.Fitness, weights=(1.0,))

# Create an individual class
# It's a list that has a fitness attribute
creator.create('Individual', list, fitness=creator.FitnessMax)

### Step 3: Create Individual and Population Generators
We register here a function that creates Individuals (possible knapsack configurations) and then a function that creates the entire population. We have to register it in our toolbox.

In [ ]:
# Register how to create an individual
# tools.initRepeat: repeat the zeroOrOne function ITEMS_LENGTH times
toolbox.register('individualCreator', tools.initRepeat, creator.Individual, toolbox.zeroOrOne, items_length)

# Register how to create a population
# tools.initRepeat: repeat individualCreator to make a list of individuals
toolbox.register('populationCreator', tools.initRepeat,list, toolbox.individualCreator,population_size)

In [ ]:
# Test: create one individual
test_individual = toolbox.individualCreator()
print(f"Chromosome: {test_individual}")
print(f"Length of the chromosome: {len(test_individual)}")
print(f"Fitness (before evaluation): {test_individual.fitness.valid}")

In [ ]:
# Test: create a population
test_population = toolbox.populationCreator()
print(f"Length of the population: {len(test_population)}")

### Step 4: Define the Fitness Function

Here we create the Fitness function and we call it `evaluate`. We register it in the toolbox.

**Remark:** The function must return a tuple (due to DEAP requirements)!

In [ ]:
def evaluate(chromosome):
    return (knapsack.getTotalValue(chromosome),)

# Register the evaluation function
toolbox.register('evaluate', evaluate)

# Test the evaluation
test_fitness = toolbox.evaluate(test_individual)
print(f"Fitness of test individual: {test_fitness[0]}")

### Step 5: Register Genetic Operators

The genetic operators can be found in the `tools` module.

Here we register the three genetic operators that you have studied during the lectures:

- Selection operator: 
  1. `selTournament` (for Tournament selection), 
  2. `selRoulette` (for RWS) 
  3. `selStochasticUniversalSampling` (for SUS). 
  
  The syntax is always:

   `toolbox.register("name_selection_method", tools.name_selection_method, arguments)`

- Crossover operator:
  1. `cxOnePoint`: one point crossover
  2. `cxTwoPoints`: two point crossover
  3. `cxUniform`: uniform crossover that modifies two sequences of individuals following a certain probability
  
- Mutation operator:
  1. `mutGaussian`: applies a Gaussian mutation following given parameters (mean, standard deviation)
  2. `mutFlipBit`: flips the value of the attributes of the input individual. It requires the probability of changing a bit.

A complete list can be found here: https://deap.readthedocs.io/en/master/api/tools.html

In [ ]:
# Selection: Tournament selection 
toolbox.register('select', tools.selTournament, tournsize=3)

# Crossover: One-point crossover 
toolbox.register('mate', tools.cxOnePoint)

# Mutation: Flip bits with probability 1/ITEMS_LENGTH per bit
toolbox.register('mutate', tools.mutFlipBit, indpb=1.0/items_length)

Let us check the crossover operator:

In [ ]:
# how crossover works
# Create two parent individuals
parent1 = toolbox.individualCreator()
parent2 = toolbox.individualCreator()

print("Before crossover:")
print(f"Parent 1: {parent1}")
print(f"Parent 2: {parent2}")

# Perform crossover
offspring1, offspring2 = toolbox.mate(parent1[:], parent2[:])

print("\nAfter crossover:")
print(f"Child 1:  {offspring1}")
print(f"Child 2:  {offspring2}")

**Exercise (in Class):** 
- Write a few line of code to test how mutation works.
- Count the number of bits that have been changed using mutation (to obtain the offspring, remember to add [0] after the function is called).
- Print the values of the fitness for the parent and child.

In [ ]:
# write your code here

### Step 6: Set Up Statistics and Hall of Fame

To analyse the result, we create a statistics that keeps track of the maximum value and the average across the generations. The Hall of Fame keeps track of the best individual in the whole generations!

In [ ]:
# Create statistics object to track fitness over generations
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register('max', np.max)   # Track maximum fitness
stats.register('avg', np.mean)  # Track average fitness

In [ ]:
# Create hall of fame to remember best individuals
hof = tools.HallOfFame(HALL_OF_FAME_SIZE)

print(f"Hall of Fame will remember top {HALL_OF_FAME_SIZE} individuals")

### Step 7: Run the Genetic Algorithm

In the final step, we run the genetic algorithm. The iterative process is already implemented in the module `algorithms` and we call the function `eaSimple`, which represents the simple genetic algorithm (see the documentation https://deap.readthedocs.io/en/master/api/algo.html#deap.algorithms.eaSimple).

In [ ]:
# Run the genetic algorithm
print("Starting evolution...\n")

population, logbook = algorithms.eaSimple(
    test_population,              # Initial population
    toolbox,                 # Toolbox with genetic operators
    cxpb=pcross,       # Crossover probability
    mutpb=pmutation,       # Mutation probability
    ngen=maxGenerations,   # Number of generations
    stats=stats,            # Statistics object
    halloffame=hof,         # Hall of fame
    verbose=True            # Print progress
)

print("\nEvolution complete!")

Let us explore in more details the results that we have derived!

In [ ]:
# Get the best individual
best_individual = hof.items[0]

knapsack.printItems(best_individual)

# Show the binary representation
print("Binary representation (1=take, 0=leave):")
print(best_individual)
print()

# Count items selected
num_items_selected = sum(best_individual)
print(f"Number of items selected: {num_items_selected} out of {len(knapsack)}")

### Exercise (in class)

1. We would like to see how well our algorithm have performed looking at the five best solutions.
Build a loop that iterates over the best five solutions saved in the hall of fame and prints: the list of selected items, the total value and the number of selected items. The first two are already implemented in the method `printItems`.

2. We would like to see how the fitness improves through the generations. Given the `maxFitnessValues` and the `meanFitnessValues`, create a plot with the two curves. Is the convergence stable?

In [ ]:
# for point 2, use the following code
maxFitnessValues, meanFitnessValues = logbook.select("max", "avg")
# plt.plot(...)
# plt.plot(...)
# ...

## Exercise (at home)

In the code introduced during the lecture, change the population size (for example, 50, 150, 300, 500). Plot the best fitness over the generations for each population size.
1. How does the population size parameter influence the speed of convergence and the quality of the result? 
2. Is there a threshold population size beyond which the quality of the solution does not improve much?

### Further Reading
- DEAP Documentation: https://deap.readthedocs.io/
- A Gentle Introduction to Genetic Algorithms with Python and Deap: https://www.youtube.com/watch?v=6h5ImIy3kpA&list=PLthTOPic19JXw2Fqtr0IwLKfSPL_kZ4E8&index=19